# RAG Document Assistant - Pipeline Notebook

**Domain:** Machine-learning study notes (6 PDF documents).
**Goal:** clean and chunk the documents, embed them, store them in a persistent vector database, build a grounded RAG pipeline with a local Ollama LLM, and evaluate it. The persisted store is exported to `backend/data/vector_store/` and loaded directly by the FastAPI backend.

**Before running**
1. `ollama pull llama3.2` and make sure the Ollama app/server is running.
2. Run **Kernel -> Restart & Run All**. The notebook is re-runnable: the vector collection is rebuilt from scratch each time.

**Sections:** 0 Setup - 2.1 Load & Inspect - 2.2 Chunking - 2.3 Embeddings & Vector Store - 2.4 Retrieval & Prompting - 2.5 Vision - 2.6 Evaluation - 2.7 Export

## 0. Setup & configuration

In [1]:
import json
import re
from pathlib import Path

import pandas as pd
from pypdf import PdfReader

# Works whether Jupyter is started from the project root or from notebooks/
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

RAW_DIR = ROOT / "data" / "raw"
STORE_DIR = ROOT / "backend" / "data" / "vector_store"   # folder loaded by the backend
CHROMA_DIR = STORE_DIR / "chroma_db"
STORE_DIR.mkdir(parents=True, exist_ok=True)

# ---- Configuration: one place, exported to config.json in section 2.7 ----
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
COLLECTION_NAME = "documents"
CHUNK_SIZE = 800       # characters
CHUNK_OVERLAP = 100    # characters
TOP_K = 4              # chunks passed to the LLM
MIN_SIMILARITY = 0.25  # below this the assistant refuses instead of guessing
LLM_MODEL = "llama3.2"

print("Project root :", ROOT)
print("Raw data dir :", RAW_DIR)
print("Vector store :", STORE_DIR)

Project root : D:\ITI\rag-assistant-project
Raw data dir : D:\ITI\rag-assistant-project\data\raw
Vector store : D:\ITI\rag-assistant-project\backend\data\vector_store


## 2.1 Load & Inspect

In [2]:
def clean_text(text: str) -> str:
    """Normalize text extracted from a PDF page."""
    text = re.sub(r"-\n(\w)", r"\1", text)   # re-join words hyphenated across lines
    text = re.sub(r"\s*\n\s*", " ", text)     # line breaks -> spaces
    text = re.sub(r"\s{2,}", " ", text)        # collapse repeated whitespace
    return text.strip()


pages, failed, needs_ocr = [], [], []
files = sorted(p for p in RAW_DIR.iterdir() if p.suffix.lower() in {".pdf", ".txt", ".md"})

for path in files:
    try:
        if path.suffix.lower() == ".pdf":
            reader = PdfReader(str(path))
            for page_no, page in enumerate(reader.pages, start=1):
                text = clean_text(page.extract_text() or "")
                if len(text) < 50:                      # almost no text -> probably a scan
                    needs_ocr.append(f"{path.name} (page {page_no})")
                    continue
                pages.append({"source": path.name, "page": page_no, "text": text})
        else:
            pages.append({"source": path.name, "page": 1, "text": clean_text(path.read_text(encoding="utf-8"))})
    except Exception as exc:
        failed.append((path.name, str(exc)))

inventory = (
    pd.DataFrame(pages)
    .groupby("source")
    .agg(pages=("page", "count"), characters=("text", lambda s: int(s.str.len().sum())))
)
display(inventory)
print(f"Files found      : {len(files)}  (formats: {sorted({p.suffix.lower() for p in files})})")
print(f"Pages extracted  : {len(pages)}")
print(f"Failed to parse  : {failed if failed else 'none'}")
print(f"Pages needing OCR: {needs_ocr if needs_ocr else 'none'}")

,pages,characters
source,,
01_supervised_learning.pdf,2,2382
02_unsupervised_learning.pdf,2,2553
03_model_evaluation.pdf,1,2260
04_overfitting_and_regularization.pdf,1,2038
05_neural_networks_basics.pdf,1,2092
06_feature_engineering.pdf,2,2149


Files found      : 6  (formats: ['.pdf'])
Pages extracted  : 9
Failed to parse  : none
Pages needing OCR: none


**Findings (for the provided sample corpus).** The corpus contains **6 PDF documents with 9 pages in total** (about 13,500 characters). All files are text-extractable, **no file failed to parse and no page needs OCR**. Cleaning needed: joining line breaks inside paragraphs and collapsing repeated whitespace. *If you replace the documents with your own, re-run this cell and update this paragraph with your numbers.*

## 2.2 Chunking strategy

In [3]:
SENTENCE_ENDS = (". ", "? ", "! ")


def chunk_text(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> list[str]:
    """Fixed-size character chunks with overlap that prefer to end at a sentence boundary."""
    chunks, start, n = [], 0, len(text)
    while start < n:
        end = min(start + chunk_size, n)
        if end < n:
            window_start = start + int(chunk_size * 0.7)          # only look in the last 30 percent
            cut = max(text.rfind(t, window_start, end) for t in SENTENCE_ENDS)
            if cut != -1:
                end = cut + 1                                     # keep the punctuation mark
            else:
                space = text.rfind(" ", window_start, end)
                if space != -1:
                    end = space
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        if end >= n:
            break
        next_start = max(end - overlap, start + 1)
        space = text.find(" ", next_start, end)                   # start the next chunk on a word boundary
        start = space + 1 if space != -1 else next_start
    return chunks


chunks = []
for page in pages:                     # chunk page by page so every chunk keeps an exact page number
    for i, piece in enumerate(chunk_text(page["text"])):
        chunks.append({
            "id": f"{Path(page['source']).stem}-p{page['page']}-c{i}",
            "source": page["source"],
            "page": page["page"],
            "text": piece,
        })

lengths = pd.Series([len(c["text"]) for c in chunks])
print(f"{len(chunks)} chunks | length min/mean/max = {lengths.min()} / {lengths.mean():.0f} / {lengths.max()} characters")
print("\nExample chunk:", chunks[1]["id"])
print(chunks[1]["text"])

24 chunks | length min/mean/max = 119 / 622 / 799 characters

Example chunk: 01_supervised_learning-p1-c1
regression fits a straight line (or a hyperplane in higher dimensions) to predict a numeric target. Logistic regression is used for binary classification and outputs a probability between 0 and 1. Decision trees split the data with a sequence of if-else questions about the features. Random forests train many decision trees on random subsets of the data and features and then average or vote on their predictions, which reduces overfitting. k-Nearest Neighbors (k-NN) predicts using the labels of the k closest training examples. Support Vector Machines (SVM) find the decision boundary that maximizes the margin between classes. Loss functions Training a supervised model means minimizing a loss function that measures prediction error.


**Why 800 characters with 100 overlap?**
- **~800 characters (about 150-200 words)** is large enough to contain a complete idea (a definition plus its example) but small enough that one chunk stays focused on one topic, which keeps the embedding precise. Much larger chunks blur several topics together; much smaller chunks split definitions from their explanations.
- **100 characters (about 12 percent) overlap** prevents a sentence that falls on a boundary from being cut in half and lost from both chunks.
- Chunks end at a **sentence boundary** when possible, and chunking is done **per page**, so every citation can point to an exact page.
- With `TOP_K = 4`, the prompt receives about 3,200 characters of context, which fits comfortably in the small local model's context window.

## 2.3 Embeddings & vector store

In [4]:
import chromadb
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer(EMBEDDING_MODEL)
texts = [c["text"] for c in chunks]
embeddings = embedder.encode(texts, batch_size=32, normalize_embeddings=True, show_progress_bar=True)
print("Embedding matrix:", embeddings.shape)

# Persistent Chroma database on disk (rebuilt from scratch on every run)
client = chromadb.PersistentClient(path=str(CHROMA_DIR))
try:
    client.delete_collection(COLLECTION_NAME)
except Exception:
    pass                                # collection did not exist yet
collection = client.create_collection(COLLECTION_NAME)   # default metric: squared L2 (cos = 1 - d/2 on unit vectors)
collection.add(
    ids=[c["id"] for c in chunks],
    embeddings=embeddings.tolist(),
    documents=texts,
    metadatas=[{"source": c["source"], "page": c["page"]} for c in chunks],
)
print("Chunks stored in Chroma:", collection.count())
print("Persisted at:", CHROMA_DIR)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\Pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:150: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Pc\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding matrix: (24, 384)


Chunks stored in Chroma: 24
Persisted at: D:\ITI\rag-assistant-project\backend\data\vector_store\chroma_db


**Choices.** `all-MiniLM-L6-v2` is a small (about 80 MB), fast sentence-embedding model that runs on CPU and works well for English semantic search. Embeddings are **normalized**, so similarity is a plain cosine score. **Chroma** persists everything to disk, so the backend just opens the folder instead of rebuilding anything.

## 2.4 Retrieval & prompting

In [5]:
def retrieve(question: str, k: int = TOP_K) -> list[dict]:
    """Return the k most similar chunks with a cosine-similarity score."""
    q = embedder.encode([question], normalize_embeddings=True).tolist()
    res = collection.query(query_embeddings=q, n_results=k, include=["documents", "metadatas", "distances"])
    return [
        {"text": doc, "source": meta["source"], "page": int(meta["page"]), "score": round(1 - dist / 2, 4)}
        for doc, meta, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0])
    ]


TEST_SET = [
    {"question": "What is the difference between classification and regression?",
     "expected_source": "01_supervised_learning.pdf", "keywords": ["continuous", "discrete", "category"]},
    {"question": "How does the elbow method help choose k in k-means?",
     "expected_source": "02_unsupervised_learning.pdf", "keywords": ["within-cluster", "inertia", "sum of squares"]},
    {"question": "What advantage does DBSCAN have over k-means?",
     "expected_source": "02_unsupervised_learning.pdf", "keywords": ["arbitrary shape", "number of clusters", "noise"]},
    {"question": "What is the F1 score?",
     "expected_source": "03_model_evaluation.pdf", "keywords": ["harmonic mean"]},
    {"question": "Why is accuracy misleading on imbalanced datasets?",
     "expected_source": "03_model_evaluation.pdf", "keywords": ["majority", "99", "always predict"]},
    {"question": "What is the difference between L1 and L2 regularization?",
     "expected_source": "04_overfitting_and_regularization.pdf", "keywords": ["sparse", "exactly zero", "squared"]},
    {"question": "What dropout rates are typical?",
     "expected_source": "04_overfitting_and_regularization.pdf", "keywords": ["0.2", "0.5"]},
    {"question": "What does the ReLU activation function do?",
     "expected_source": "05_neural_networks_basics.pdf", "keywords": ["max(0", "zero", "negative"]},
    {"question": "What is data leakage and how can I avoid it when scaling features?",
     "expected_source": "06_feature_engineering.pdf", "keywords": ["training set", "split"]},
    {"question": "How should a categorical feature such as color be encoded?",
     "expected_source": "06_feature_engineering.pdf", "keywords": ["one-hot"]},
    # Out-of-scope questions: the assistant must refuse instead of answering from its own knowledge
    {"question": "Who won the 2018 FIFA World Cup?", "expected_source": None, "keywords": []},
    {"question": "What is the capital of Australia?", "expected_source": None, "keywords": []},
]

rows = []
for t in TEST_SET:
    hits = retrieve(t["question"])
    top = hits[0]
    in_scope = t["expected_source"] is not None
    rows.append({
        "question": t["question"],
        "top_source": f"{top['source']} (p.{top['page']})",
        "top_score": top["score"],
        "expected": t["expected_source"] or "(out of scope)",
        "relevant_in_top_k": (t["expected_source"] in {h["source"] for h in hits}) if in_scope else None,
        "would_be_refused": top["score"] < MIN_SIMILARITY,
    })
retrieval_df = pd.DataFrame(rows)
pd.set_option("display.max_colwidth", 70)
display(retrieval_df)
print("Relevant source in top-k:", int(retrieval_df["relevant_in_top_k"].dropna().sum()), "/", int(retrieval_df["relevant_in_top_k"].notna().sum()))
print("Check the top_score of the out-of-scope questions and tune MIN_SIMILARITY if needed.")

,question,top_source,top_score,expected,relevant_in_top_k,would_be_refused
0,What is the difference between classification and regression?,01_supervised_learning.pdf (p.1),0.5944,01_supervised_learning.pdf,True,False
1,How does the elbow method help choose k in k-means?,02_unsupervised_learning.pdf (p.1),0.6381,02_unsupervised_learning.pdf,True,False
2,What advantage does DBSCAN have over k-means?,02_unsupervised_learning.pdf (p.1),0.7528,02_unsupervised_learning.pdf,True,False
3,What is the F1 score?,03_model_evaluation.pdf (p.1),0.4160,03_model_evaluation.pdf,True,False
4,Why is accuracy misleading on imbalanced datasets?,03_model_evaluation.pdf (p.1),0.4679,03_model_evaluation.pdf,True,False
5,What is the difference between L1 and L2 regularization?,04_overfitting_and_regularization.pdf (p.1),0.5100,04_overfitting_and_regularization.pdf,True,False
6,What dropout rates are typical?,04_overfitting_and_regularization.pdf (p.1),0.4691,04_overfitting_and_regularization.pdf,True,False
7,What does the ReLU activation function do?,05_neural_networks_basics.pdf (p.1),0.3993,05_neural_networks_basics.pdf,True,False
8,What is data leakage and how can I avoid it when scaling features?,06_feature_engineering.pdf (p.1),0.5660,06_feature_engineering.pdf,True,False
9,How should a categorical feature such as color be encoded?,06_feature_engineering.pdf (p.1),0.5259,06_feature_engineering.pdf,True,False


Relevant source in top-k: 10 / 10
Check the top_score of the out-of-scope questions and tune MIN_SIMILARITY if needed.


In [6]:
REFUSAL = "I don't have enough information in the provided documents to answer that."

SYSTEM_PROMPT = (
    "You are a document assistant. Answer the user's question using ONLY the numbered "
    "context passages provided. Cite the passages you use with their numbers in square "
    "brackets, like [1] or [2][3]. If the context does not contain the answer, reply "
    f'exactly: "{REFUSAL}" Never use outside knowledge. '
    "Keep the answer concise (at most 5 sentences)."
)


def build_messages(question: str, hits: list[dict]) -> list[dict]:
    context = "\n\n".join(
        f"[{i}] (source: {h['source']}, page {h['page']})\n{h['text']}" for i, h in enumerate(hits, 1)
    )
    user = f"Context:\n{context}\n\nQuestion: {question}\n\nAnswer (with [n] citations):"
    return [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user}]


def extract_citations(answer: str, n_chunks: int) -> list[int]:
    numbers = set()
    for group in re.findall(r"\[([\d,\s]+)\]", answer):
        for part in group.split(","):
            if part.strip().isdigit() and 1 <= int(part) <= n_chunks:
                numbers.add(int(part))
    return sorted(numbers)


import ollama


def ask(question: str, k: int = TOP_K, min_score: float = MIN_SIMILARITY) -> dict:
    """Full RAG: retrieve -> (refuse if nothing relevant) -> prompt -> generate -> cite."""
    hits = [h for h in retrieve(question, k) if h["score"] >= min_score]
    if not hits:
        return {"answer": REFUSAL, "sources": [], "hits": []}
    response = ollama.chat(model=LLM_MODEL, messages=build_messages(question, hits), options={"temperature": 0.1})
    answer = response["message"]["content"].strip()
    if REFUSAL in answer:
        return {"answer": answer, "sources": [], "hits": hits}
    used = [hits[i - 1] for i in extract_citations(answer, len(hits))] or hits
    sources = list(dict.fromkeys(f"{h['source']} (page {h['page']})" for h in used))
    return {"answer": answer, "sources": sources, "hits": hits}


demo = ask("What is the F1 score?")
print(demo["answer"])
print("Sources:", demo["sources"])

The F1 score is the harmonic mean of precision and recall, computed as 2PR divided by (P + R). [1]
Sources: ['03_model_evaluation.pdf (page 1)']


**Grounding design.** Three layers keep the assistant from answering out of its own memory: (1) the prompt says to use *only* the numbered passages and to reply with a fixed refusal sentence otherwise, (2) a **similarity threshold** skips the LLM entirely when nothing relevant is retrieved, and (3) a low **temperature (0.1)** reduces creative drift. Citations are the passage numbers `[n]` produced by the model, mapped back to `file (page n)` in code.

## 2.5 Vision component

Not applicable: this project follows the **Core Track** (text-only RAG). No image dataset or YOLO model is used.

## 2.6 Evaluation

In [7]:
records = []
for t in TEST_SET:
    result = ask(t["question"])
    answer = result["answer"]
    in_scope = t["expected_source"] is not None
    refused = REFUSAL in answer
    retrieved = {h["source"] for h in result["hits"]}

    if in_scope:
        relevant = t["expected_source"] in retrieved
        cites_expected = any(s.startswith(t["expected_source"]) for s in result["sources"])
        has_keyword = any(k.lower() in answer.lower() for k in t["keywords"])
        verdict = "correct" if (relevant and cites_expected and has_keyword and not refused) else "check manually"
    else:
        relevant = None
        verdict = "correct (refused)" if refused else "hallucinated?"

    records.append({
        "question": t["question"],
        "retrieved_source": ", ".join(sorted(retrieved)) or "-",
        "context_relevant": relevant,
        "answer": answer,
        "cited_sources": "; ".join(result["sources"]) or "-",
        "auto_verdict": verdict,
    })

results_df = pd.DataFrame(records)
pd.set_option("display.max_colwidth", 120)
display(results_df[["question", "retrieved_source", "context_relevant", "cited_sources", "auto_verdict"]])
results_df.to_csv(ROOT / "notebooks" / "evaluation_results.csv", index=False)

n_ok = results_df["auto_verdict"].str.startswith("correct").sum()
print(f"Auto-verdict correct: {n_ok} / {len(results_df)}  (read the full answers below and confirm by hand)")
for r in records:
    print("\nQ:", r["question"], "\nA:", r["answer"], "\nSources:", r["cited_sources"], "| verdict:", r["auto_verdict"])

,question,retrieved_source,context_relevant,cited_sources,auto_verdict
0,What is the difference between classification and regression?,"01_supervised_learning.pdf, 03_model_evaluation.pdf",True,01_supervised_learning.pdf (page 1),correct
1,How does the elbow method help choose k in k-means?,"02_unsupervised_learning.pdf, 03_model_evaluation.pdf",True,02_unsupervised_learning.pdf (page 1),correct
2,What advantage does DBSCAN have over k-means?,02_unsupervised_learning.pdf,True,02_unsupervised_learning.pdf (page 1),correct
3,What is the F1 score?,03_model_evaluation.pdf,True,03_model_evaluation.pdf (page 1),correct
4,Why is accuracy misleading on imbalanced datasets?,"03_model_evaluation.pdf, 06_feature_engineering.pdf",True,03_model_evaluation.pdf (page 1),correct
5,What is the difference between L1 and L2 regularization?,"04_overfitting_and_regularization.pdf, 06_feature_engineering.pdf",True,04_overfitting_and_regularization.pdf (page 1); 06_feature_engineering.pdf (page 2),correct
6,What dropout rates are typical?,"01_supervised_learning.pdf, 03_model_evaluation.pdf, 04_overfitting_and_regularization.pdf",True,04_overfitting_and_regularization.pdf (page 1),correct
7,What does the ReLU activation function do?,"04_overfitting_and_regularization.pdf, 05_neural_networks_basics.pdf",True,05_neural_networks_basics.pdf (page 1),correct
8,What is data leakage and how can I avoid it when scaling features?,"04_overfitting_and_regularization.pdf, 06_feature_engineering.pdf",True,06_feature_engineering.pdf (page 1),correct
9,How should a categorical feature such as color be encoded?,06_feature_engineering.pdf,True,06_feature_engineering.pdf (page 1),correct


Auto-verdict correct: 12 / 12  (read the full answers below and confirm by hand)

Q: What is the difference between classification and regression? 
A: Classification predicts a discrete category, whereas regression predicts a continuous numeric value. The main difference between the two is the type of target the model has to predict. Classification predicts a categorical value, such as spam or not spam, whereas regression predicts a continuous value, such as the price of a house. Both use labeled data, but the type of target is the key difference between the two. [1][2] 
Sources: 01_supervised_learning.pdf (page 1) | verdict: correct

Q: How does the elbow method help choose k in k-means? 
A: The elbow method helps choose k in k-means by plotting the within-cluster sum of squares (also called inertia) against k. The best k is the point where the curve bends like an elbow and the improvement slows down sharply. [1] This is because the inertia represents the amount of variation within ea

### Failure analysis *(complete this cell after running the notebook)*

The automatic verdict only checks: relevant source retrieved, that source cited, an expected keyword present, and refusal on out-of-scope questions. **Read every answer and correct the verdicts by hand**, then replace this cell with your own observations. Things worth looking for:

- **Topic overlap between documents** (e.g. L1 regularization appears in both the regularization and the feature-engineering notes): does the right document rank first?
- **Weak citations:** does the small model forget the `[n]` markers or cite the wrong passage?
- **Out-of-scope questions:** were they refused, or did the model answer from its own knowledge? If a refusal was missed, raise `MIN_SIMILARITY`.

**Mitigations already built in:** strict "context only" system prompt, fixed refusal sentence, similarity threshold, temperature 0.1, and per-page chunks for precise citations.

## 2.7 Export for the backend

In [8]:
config = {
    "embedding_model": EMBEDDING_MODEL,
    "collection_name": COLLECTION_NAME,
    "chunk_size": CHUNK_SIZE,
    "chunk_overlap": CHUNK_OVERLAP,
    "similarity": "squared L2 on normalized embeddings (cosine = 1 - d/2)",
    "num_chunks": collection.count(),
    "source_files": sorted({c["source"] for c in chunks}),
    "recommended_top_k": TOP_K,
    "recommended_min_similarity": MIN_SIMILARITY,
    "recommended_llm": LLM_MODEL,
}
(STORE_DIR / "config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")

# Verify exactly like the backend does: open the persisted store from disk
reopened = chromadb.PersistentClient(path=str(CHROMA_DIR)).get_collection(COLLECTION_NAME)
assert reopened.count() == len(chunks), "persisted store does not match the notebook state"
print("Exported to:", STORE_DIR)
print(json.dumps(config, indent=2))

Exported to: D:\ITI\rag-assistant-project\backend\data\vector_store
{
  "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
  "collection_name": "documents",
  "chunk_size": 800,
  "chunk_overlap": 100,
  "similarity": "squared L2 on normalized embeddings (cosine = 1 - d/2)",
  "num_chunks": 24,
  "source_files": [
    "01_supervised_learning.pdf",
    "02_unsupervised_learning.pdf",
    "03_model_evaluation.pdf",
    "04_overfitting_and_regularization.pdf",
    "05_neural_networks_basics.pdf",
    "06_feature_engineering.pdf"
  ],
  "recommended_top_k": 4,
  "recommended_min_similarity": 0.25,
  "recommended_llm": "llama3.2"
}
